# unity-lab — v0 SYSTEM PROTOTYPE (rough; hardening later)

The workstation loop in one notebook: **boot pinned Unity editor → activate license → sync a project from Drive → build (Linux or WebGL) → publish artifacts to Drive.**

- VM is **disposable**: nothing here is permanent. Source of truth = your Drive folders + the agent's repo (wired later).
- Editor tarballs are **cached on Drive** after first fetch, so later boots skip the 4.5 GB CDN download.
- License auto-activates per fresh VM (Unity Personal entitlement); ~monthly re-check.

## Before you run
1. Runtime > Change runtime type > **CPU**.
2. GATE "license" cell: fill `UNAME` / `UPASS` placeholders **locally** (never paste into chat).
3. Run all. Drive popup once (cache + project + artifacts).
4. Optional Drive folders (auto-created where possible):
   - `MyDrive/unity-lab/cache/` — editor tarballs land here on first fetch
   - `MyDrive/unity-lab/projects/<name>/Assets/...` — put a Unity project here to build it (else a demo project is scaffolded)
   - `MyDrive/unity-lab/builds/<timestamp>/` — build outputs land here

The demo project auto-authors **Cube Dodger** — a code-only endless runner (GameAuthor pattern: scripts written by the cell, game boots in the player at runtime). Play built WebGL via the tunnel cell the agent gives you.

Cells are self-sufficient and print PASS/FAIL + MEASURE lines. Rough intent: correctness first, polish later.


### 🧰 1 · Setup
**What this does:** installs tools, mounts Drive, shows the run form (cache / sync / build target). Pick CPU runtime first.

In [ ]:
#@title unity-lab v0 — init (runtime must be CPU) { display-mode: "form" }
ENABLE_DRIVE_CACHE = True #@param {type:"boolean"}
SYNC_PROJECT = True #@param {type:"boolean"}
BUILD_TARGET = "WebGL" #@param ['Linux', 'WebGL', 'Windows', 'Android', 'Mac']
GAME = "runandgun" #@param ['runandgun', 'demo']
KEEP_ARTIFACTS_ALSO_LOCAL = True #@param {type:"boolean"}

# --- init: tools + Drive + cache dir ---
import subprocess
r = subprocess.run(['bash', '-lc', 'apt-get -qq update >/dev/null 2>&1; apt-get install -y -qq aria2 pigz xz-utils time zip >/dev/null 2>&1; echo APT_EXIT=$?'], capture_output=True, text=True)
print('apt:', (r.stdout or '').strip(), (r.stderr or '').strip()[-500:])
assert 'APT_EXIT=0' in r.stdout, 'apt failed'
r = subprocess.run(['bash', '-lc', 'echo == VM spec ==; nproc; free -m | head -2; df -h /content | tail -1; head -1 /etc/os-release'])
ALSA = subprocess.run(['bash', '-lc',
    'mkdir -p /tmp/xdg && chmod 700 /tmp/xdg; '
    'printf "pcm.!default {\n  type null\n}\nctl.!default {\n  type null\n}\n" > /etc/asound.conf; echo ALSA_XDG_OK'],
    capture_output=True, text=True)
print('audio/headless fix:', ALSA.stdout.strip())
if ENABLE_DRIVE_CACHE or SYNC_PROJECT:
    from google.colab import drive
    drive.mount('/content/drive')
    import os
    for sub in ['cache', 'projects', 'builds']:
        os.makedirs('/content/drive/MyDrive/unity-lab/' + sub, exist_ok=True)
    print('DRIVE_READY: MyDrive/unity-lab/{cache,projects,builds} ensured')
print('FORM:', 'ENABLE_DRIVE_CACHE=%s' % ENABLE_DRIVE_CACHE, '| SYNC_PROJECT=%s' % SYNC_PROJECT, '| BUILD_TARGET=%s' % BUILD_TARGET, '| GAME=%s' % GAME)


### 📌 2 · Config
**What this does:** writes the pinned-Unity config JSON that every later cell reads back.

In [ ]:
import json, os
# --- config: pinned editor + cache paths (derived from the form globals of cell 1) ---
CFG = {
    'VERSION': '6000.3.10f1',
    'REV': 'e35f0c77bd8e',
    'CMD5': 'd575da705f9f8d9d6bc4bf70546eca04',
    'EDITOR_URL': 'https://download.unity3d.com/download_unity/e35f0c77bd8e/LinuxEditorInstaller/Unity.tar.xz',
    'WEBGL_URL': 'https://download.unity3d.com/download_unity/e35f0c77bd8e/LinuxEditorTargetInstaller/UnitySetup-WebGL-Support-for-Editor-6000.3.10f1.tar.xz',
    'WEBGL_MD5': '74706376db4dd2202532a0d71824a793',
    'MAC_URL': 'https://download.unity3d.com/download_unity/e35f0c77bd8e/MacEditorTargetInstaller/UnitySetup-Mac-Mono-Support-for-Editor-6000.3.10f1.pkg',
    'MAC_MD5': 'f7b992c77a2dfe976e501fb6487b41b2',
    'WINDOWS_URL': 'https://download.unity3d.com/download_unity/e35f0c77bd8e/MacEditorTargetInstaller/UnitySetup-Windows-Mono-Support-for-Editor-6000.3.10f1.pkg',
    'WINDOWS_MD5': '67f2894f459ca1f3502a85b2257328e1',
    'ANDROID_URL': 'https://download.unity3d.com/download_unity/e35f0c77bd8e/MacEditorTargetInstaller/UnitySetup-Android-Support-for-Editor-6000.3.10f1.pkg',
    'ANDROID_MD5': '69cb13632b3e5f5ef95a88ec3b670b16',
    'ANDROID_JDK_URL': 'https://download.unity3d.com/download_unity/open-jdk/open-jdk-linux-x64/jdk17.0.9-9_8d1cbcce56285f3146cf7761353a643fe573b39e45bd94f35590dca39277f667.zip',
    # (url, md5 or '', unzip destination relative to /content/ue, rename_from, rename_to)
    'ANDROID_CHAIN': [
        ['sdk-tools', 'https://dl.google.com/android/repository/sdk-tools-linux-4333796.zip', '', 'AndroidPlayer/SDK', '', '', 'AndroidPlayer/SDK/tools'],
        ['platform-tools', 'https://dl.google.com/android/repository/platform-tools_r34.0.5-linux.zip', '', 'AndroidPlayer/SDK', '', '', 'AndroidPlayer/SDK/platform-tools'],
        ['build-tools', 'https://dl.google.com/android/repository/build-tools_r34-linux.zip', '', 'AndroidPlayer/SDK/build-tools', 'android-14', '34.0.0', 'AndroidPlayer/SDK/build-tools/34.0.0'],
        ['platform-33', 'https://dl.google.com/android/repository/platform-33_r02.zip', '', 'AndroidPlayer/SDK/platforms', 'android-13', 'android-33', 'AndroidPlayer/SDK/platforms/android-33'],
        ['platform-34', 'https://dl.google.com/android/repository/platform-34-ext7_r02.zip', '', 'AndroidPlayer/SDK/platforms', '', '', 'AndroidPlayer/SDK/platforms/android-34'],
        ['ndk-r27c', 'https://dl.google.com/android/repository/android-ndk-r27c-linux.zip', '', 'AndroidPlayer', 'android-ndk-r27c', 'NDK', 'AndroidPlayer/NDK'],
        ['cmdline-tools', 'https://dl.google.com/android/repository/commandlinetools-linux-8092744_latest.zip', '', 'AndroidPlayer/SDK/cmdline-tools', 'cmdline-tools', '6.0', 'AndroidPlayer/SDK/cmdline-tools/6.0'],
        ['cmake', 'https://dl.google.com/android/repository/cmake-3.22.1-linux.zip', '', 'AndroidPlayer/SDK/cmake', 'cmake', '3.22.1', 'AndroidPlayer/SDK/cmake/3.22.1'],
    ],
    'ENABLE_DRIVE_CACHE': bool(globals().get('ENABLE_DRIVE_CACHE', True)),
    'SYNC_PROJECT': bool(globals().get('SYNC_PROJECT', True)),
    'BUILD_TARGET': globals().get('BUILD_TARGET', 'WebGL'),
    'GAME': globals().get('GAME', 'runandgun'),
    'DRIVE_CACHE': '/content/drive/MyDrive/unity-lab/cache',
    'DRIVE_PROJECTS': '/content/drive/MyDrive/unity-lab/projects',
    'DRIVE_BUILDS': '/content/drive/MyDrive/unity-lab/builds',
}
json.dump(CFG, open('/content/unitylab_config.json', 'w'))
print('Pinned:', CFG['VERSION'], '| rev:', CFG['REV'], '| target:', CFG['BUILD_TARGET'], '| game:', CFG['GAME'], '| cache:', CFG['ENABLE_DRIVE_CACHE'])


### 🔄 3 · Keep-alive
**What this does:** best-effort anti-tab-sleep: silent audio + heartbeat. Keep the tab open during long builds.

In [ ]:
from IPython.display import Javascript, display
# --- keep-alive (best-effort: silent WebAudio + heartbeat) ---
display(Javascript('''
(function(){
  if (window.__unitylab_ka) { console.log('keepalive already running'); return; }
  window.__unitylab_ka = true;
  try {
    var ctx = new (window.AudioContext || window.webkitAudioContext)();
    var o = ctx.createOscillator(); var g = ctx.createGain();
    g.gain.value = 0.0; o.connect(g); g.connect(ctx.destination); o.start();
    if (ctx.state === 'suspended' && ctx.resume) { ctx.resume(); }
  } catch(e) { console.log('keepalive audio unavailable:', e); }
  setInterval(function(){ console.log('unity-lab keepalive ' + new Date().toISOString()); }, 60000);
  console.log('keepalive started');
})();
'''))
print('KEEPALIVE_DISPLAYED (tab must still stay open)')


### 🏇 4 · Boot editor
**What this does:** boots the pinned Unity editor: Drive-cached tarball first (md5-checked), CDN fallback with stash-back-to-cache. Skips if already extracted.

In [ ]:
import subprocess, os, time, glob, json, shutil
# --- boot pinned editor from Drive cache (fallback CDN) --- PROVEN pipeline + cache-first ---
cfg = json.load(open('/content/unitylab_config.json'))
t0 = time.time()
TGZ = '/content/Unity.tar.xz'

def checksum(path):
    import hashlib
    m = hashlib.md5()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(1 << 22), b''):
            m.update(chunk)
    return m.hexdigest()

need_download = True
if os.path.exists(TGZ) and checksum(TGZ) == cfg['CMD5']:
    print('LOCAL TARBALL OK (md5) — reuse')
    need_download = False
cache_tgz = os.path.join(cfg['DRIVE_CACHE'], 'Unity.tar.xz') if cfg['ENABLE_DRIVE_CACHE'] else None
if need_download and cache_tgz and os.path.exists(cache_tgz):
    print('Drive cache HIT — copying 4.5 GB (measuring throughput) ...')
    tc = time.time(); sz = os.path.getsize(cache_tgz)
    subprocess.run(['bash', '-lc', 'cp "%s" /content/Unity.tar.xz' % cache_tgz])
    dt = max(time.time() - tc, 0.001)
    print('MEASURE: drive-copy %d MB in %.1f s = %.1f MB/s' % (sz / 1048576.0, dt, (sz / 1048576.0) / dt))
    if checksum(TGZ) == cfg['CMD5']:
        print('CACHE TARBALL VERIFIED — no CDN needed')
        need_download = False
    else:
        print('CACHE TARBALL CORRUPT — falling back to CDN'); os.remove(TGZ)
if need_download:
    r = subprocess.run(['bash', '-lc', 'aria2c -x8 -s8 --console-log-level=warn -d /content -o Unity.tar.xz "%s"' % cfg['EDITOR_URL']],
                       capture_output=True, text=True, timeout=1800)
    print('CDN download rc:', r.returncode)
    if (r.stdout or '').strip(): print((r.stdout or '')[-600:])
    if r.returncode != 0: print((r.stderr or '')[-2000:])
    ok = checksum(TGZ) == cfg['CMD5'] if os.path.exists(TGZ) else False
    assert ok, 'MD5_MISMATCH after download'
    if cache_tgz:
        try:
            shutil.copyfile(TGZ, cache_tgz)
            print('STASHED tarball to Drive cache for future boots')
        except Exception as e:
            print('cache stash best-effort failed (fine):', repr(e))

ubin = '/content/unity_bin.txt'
already = False
if os.path.exists(ubin):
    p = open(ubin).read().strip()
    if p and os.path.exists(p):
        miss = subprocess.run(['bash', '-lc', 'ldd "%s" | grep "not found" || true' % p], capture_output=True, text=True).stdout.strip()
        already = (not miss)
if already:
    print('GATE 1 SKIP — editor already extracted at:', p)
else:
    te = time.time()
    r = subprocess.run(['bash', '-lc', 'tar -I "xz -T0" -xf /content/Unity.tar.xz -C /content/ue || (mkdir -p /content/ue && tar -xJf /content/Unity.tar.xz -C /content/ue); echo TAR_RC=$?'],
                       capture_output=True, text=True, timeout=3600)
    print('MEASURE: extract %.1f min (xz -T0 attempt, fallback xJf)' % ((time.time() - te) / 60))
    if r.returncode != 0: print((r.stderr or '')[-2000:])
    assert r.returncode == 0, 'extract FAILED'
    cands = glob.glob('/content/ue/*/Editor/Unity') + glob.glob('/content/ue/Editor/Unity')
    assert cands, 'editor binary not found after extract'
    UNITY_BIN = cands[0]
    open(ubin, 'w').write(UNITY_BIN)
    print('editor: %.1f min total to boot' % ((time.time() - t0) / 60))
UNITY_BIN = open(ubin).read().strip()
# --- editor runtime dependencies (fresh VMs: Unity cannot exec without X libs) ---
DEPS = 'libx11-6 libx11-xcb1 libxext6 libxrandr2 libxinerama1 libxcursor1 libxi6 libfontconfig1 libglib2.0-0 libnss3 libdbus-1-3 libxcomposite1 libxdamage1 libxfixes3 libxss1 libxtst6 libdrm2 libgbm1 libgtk-3-0 libglu1-mesa libsm6 libice6 libpng16-16'
T64 = 'libasound2t64 libatk1.0-0t64 libatk-bridge2.0-0t64 libcups2t64'
apt = subprocess.run(['bash', '-lc', 'apt-get update -qq && (apt-get install -y -qq ' + DEPS + ' ' + T64 + ' || apt-get install -y -qq ' + DEPS + ' libasound2 libatk1.0-0 libatk-bridge2.0-0 libcups2)'],
                     capture_output=True, text=True, timeout=900)
print('deps install rc:', apt.returncode)
if apt.returncode != 0:
    print('--- APT STDERR (tail) ---'); print((apt.stderr or '(empty)')[-2000:])
miss = subprocess.run(['bash', '-lc', 'ldd "%s" | grep "not found" || true' % UNITY_BIN], capture_output=True, text=True).stdout.strip()
print('loader-missing libs:', miss if miss else '(none)')
assert not miss, 'still missing shared libs: ' + miss
print('GATE 1 PASS — binary:', UNITY_BIN)
subprocess.run(['bash', '-lc', '"%s" -version' % UNITY_BIN])


### 🔑 5 · License
**What this does:** one-shot Unity Personal activation — fill UNAME/UPASS placeholders first; auto-skips if this VM is already licensed.

In [ ]:
import subprocess, os
# --- license ACTIVATION --- (proven recipe; SKIPS silently if this VM is already licensed) ---
# Credentials are prompted ONLY when activation is actually needed — never stored in the notebook, never pasted into chat.
# If output shows code 132.182: wait 24h before any retry (retries extend the block).

UNITY = open('/content/unity_bin.txt').read().strip()
UCLIENT = UNITY.replace('/Editor/Unity', '') + '/Editor/Data/Resources/Licensing/Client/Unity.Licensing.Client'
assert os.path.exists(UCLIENT), 'licensing client missing: ' + UCLIENT

def smoke():
    r = subprocess.run([UNITY, '-batchmode', '-quit', '-nographics', '-logFile', '/content/lic_smoke.log'],
                       capture_output=True, text=True, timeout=900)
    liclog = open('/content/lic_smoke.log', errors='ignore').read() if os.path.exists('/content/lic_smoke.log') else ''
    BAD = [s for s in ['no valid Unity Editor license', 'was not found', 'entitlement'] if s.lower() in liclog.lower()]
    return r.returncode, BAD, liclog

skip = False
lic_xml = os.path.expanduser('~/.config/unity3d/Unity/licenses/UnityEntitlementLicense.xml')
if os.path.exists(lic_xml):
    rc0, bad0, _ = smoke()
    if rc0 == 0:
        skip = True
        print('GATE 2 SKIP — license already active on this VM (smoke rc 0)')

if not skip:
    import getpass
    UNAME = input('Unity ID email: ')
    UPASS = getpass.getpass('Unity ID password (hidden input): ')
    rc1 = subprocess.run([UCLIENT, '--show-seats', '--username', UNAME, '--password', UPASS], capture_output=True, text=True, timeout=600)
    print('SHOW_SEATS rc:', rc1.returncode); print(((rc1.stdout or '') + (rc1.stderr or ''))[-1500:])
    assert rc1.returncode == 0, 'CREDENTIAL_PRECHECK_FAILED (143.002 = wrong password; 132.182 = blocked, wait 24h)'
    rc2 = subprocess.run([UCLIENT, '--activate-all', '--include-personal', '--username', UNAME, '--password', UPASS], capture_output=True, text=True, timeout=600)
    print('ACTIVATE rc:', rc2.returncode); print(((rc2.stdout or '') + (rc2.stderr or ''))[-1500:])
    assert rc2.returncode == 0, 'ACTIVATION FAILED'

rc3, bad, liclog = smoke()
print('EDITOR_SMOKE rc:', rc3, '| bad keywords:', bad or 'none')
assert rc3 == 0, 'GATE 2 FAIL - smoke exit %d\n%s' % (rc3, '\n'.join(liclog.splitlines()[-40:]))
print('GATE 2 PASS — license accepted headlessly')


### 📦 6 · Sync project
**What this does:** pulls your Unity project from Drive (or scaffolds a demo project if none found).

In [ ]:
import subprocess, os, glob, json, time
# --- sync project from Drive (or scaffold a demo one) ---
cfg = json.load(open('/content/unitylab_config.json'))
CONTRA_FILES = {}
CONTRA_FILES['BuildScript.cs'] = '''using System.IO; using System.Linq; using UnityEngine; using UnityEditor; using UnityEditor.SceneManagement;
public static class BuildScript {
    [MenuItem("Build/Perform")]
    public static void PerformBuild() {
        var scenes = new string[] { "Assets/Scenes/Sample.unity" };
        if (!File.Exists(scenes[0])) {
            var scene = EditorSceneManager.NewScene(NewSceneSetup.EmptyScene, NewSceneMode.Single);
            System.IO.Directory.CreateDirectory("Assets/Scenes");
            EditorSceneManager.SaveScene(scene, scenes[0]);
            Debug.Log("SCENE_CREATED_BY_SCRIPT");
        }
        EditorBuildSettings.scenes = scenes.Select(s => new EditorBuildSettingsScene(s, true)).ToArray();
        AssetDatabase.Refresh(ImportAssetOptions.ForceUpdate);
        AssetDatabase.SaveAssets();
        if (!File.Exists(scenes[0])) throw new System.Exception("SCENE_SAVE_FAILED: " + scenes[0]);
        var target = BuildTarget.WebGL; var outDir = "Builds/WebGL/probe";
        if (target == BuildTarget.WebGL) PlayerSettings.WebGL.compressionFormat = WebGLCompressionFormat.Disabled;
        var report = BuildPipeline.BuildPlayer(scenes, outDir, target, BuildOptions.None);
        if (report.summary.result != UnityEditor.Build.Reporting.BuildResult.Succeeded) throw new System.Exception("BUILD_FAILED: " + report.summary.result);
        Debug.Log("BUILD_OK " + report.summary.totalSize + " bytes");
    }
}
'''
CONTRA_FILES['Bullet2D.cs'] = '''using UnityEngine;

public class Bullet2D : MonoBehaviour {
    public Collider2D ignore;
    void Start() {
        Destroy(gameObject, 2f);
    }
    void OnTriggerEnter2D(Collider2D other) {
        if (other == ignore) return;
        Destroy(gameObject);
    }
}
'''
CONTRA_FILES['ContraBoot.cs'] = '''using UnityEngine;
using UnityEngine.UI;

public static class GameSprites {
    public static Sprite White;
    public static Sprite Get() {
        if (White == null) {
            var tex = new Texture2D(4, 4, TextureFormat.RGBA32, false);
            var cols = new Color32[16];
            for (int i = 0; i < 16; i++) cols[i] = new Color32(255, 255, 255, 255);
            tex.SetPixels32(cols);
            tex.Apply();
            White = Sprite.Create(tex, new Rect(0f, 0f, 4f, 4f), new Vector2(0.5f, 0.5f), 4f);
            White.name = "WhiteSquare";
        }
        return White;
    }
}

public static class ContraBoot {
    [RuntimeInitializeOnLoadMethod(RuntimeInitializeLoadType.AfterSceneLoad)]
    static void Boot() {
        var stale = GameObject.Find("Cube");
        if (stale != null) Object.Destroy(stale);
        if (GameObject.Find("ContraGame") != null) return;
        new GameObject("ContraGame");

        var camGo = new GameObject("MainCamera");
        var cam = camGo.AddComponent<Camera>();
        cam.tag = "MainCamera";
        cam.orthographic = true;
        cam.orthographicSize = 6f;
        cam.clearFlags = CameraClearFlags.SolidColor;
        cam.backgroundColor = new Color(0.08f, 0.09f, 0.16f);
        camGo.transform.position = new Vector3(0f, 1f, -10f);
        var fc = camGo.AddComponent<FollowCam>();
        fc.minX = -26f; fc.maxX = 26f;

        MakeBox("Ground", new Vector3(0f, -3.5f, 0f), new Vector3(80f, 1f, 1f), new Color(0.34f, 0.36f, 0.40f));
        MakeBox("WallL", new Vector3(-40f, 1f, 0f), new Vector3(1f, 14f, 1f), new Color(0f, 0f, 0f, 0f));
        MakeBox("WallR", new Vector3(40f, 1f, 0f), new Vector3(1f, 14f, 1f), new Color(0f, 0f, 0f, 0f));

        var p = new GameObject("Player");
        p.transform.position = new Vector3(0f, -1.9f, 0f);
        var psr = p.AddComponent<SpriteRenderer>();
        psr.sprite = GameSprites.Get();
        psr.color = new Color(0.25f, 0.55f, 1f);
        p.transform.localScale = new Vector3(1f, 2f, 1f);
        p.AddComponent<BoxCollider2D>();
        p.AddComponent<Rigidbody2D>();
        p.AddComponent<ContraPlayer>();
        fc.target = p.transform;

        MakeHint();
        Debug.Log("CONTRA_BOOTED");
    }

    static void MakeBox(string name, Vector3 pos, Vector3 size, Color color) {
        var go = new GameObject(name);
        go.transform.position = pos;
        go.transform.localScale = size;
        if (color.a > 0f) {
            var sr = go.AddComponent<SpriteRenderer>();
            sr.sprite = GameSprites.Get();
            sr.color = color;
        }
        go.AddComponent<BoxCollider2D>();
    }

    static void MakeHint() {
        var canvasGo = new GameObject("HintCanvas");
        var canvas = canvasGo.AddComponent<Canvas>();
        canvas.renderMode = RenderMode.ScreenSpaceOverlay;
        var txt = canvasGo.AddComponent<Text>();
        txt.color = new Color(1f, 1f, 1f, 0.85f);
        txt.fontSize = 18;
        txt.alignment = TextAnchor.LowerLeft;
        txt.text = "A/D move   Space jump   W/S aim   hold Shift = aim lock   Z fire";
        Font f = null;
        try { f = Resources.GetBuiltinResource<Font>("LegacyRuntime.ttf"); } catch (System.Exception) {}
        if (f == null) { try { f = Resources.GetBuiltinResource<Font>("Arial.ttf"); } catch (System.Exception) {} }
        txt.font = f;
        var rt = txt.rectTransform;
        rt.anchorMin = new Vector2(0f, 0f);
        rt.anchorMax = new Vector2(1f, 0f);
        rt.offsetMin = new Vector2(12f, 8f);
        rt.offsetMax = new Vector2(-12f, 44f);
    }
}
'''
CONTRA_FILES['ContraPlayer.cs'] = '''using UnityEngine;

public class ContraPlayer : MonoBehaviour {
    public float runSpeed = 7f;
    public float jumpSpeed = 13f;
    public float bulletSpeed = 14f;
    public float fireCooldown = 0.16f;
    public float barrelDist = 0.75f;

    Rigidbody2D rb;
    Collider2D col;
    Transform barrel;
    float fireTimer;
    float faceDir = 1f;
    Vector2 aimDir = Vector2.right;
    bool grounded;

    void Awake() {
        rb = GetComponent<Rigidbody2D>();
        col = GetComponent<Collider2D>();
        rb.gravityScale = 4f;
        rb.freezeRotation = true;
        var b = new GameObject("Barrel");
        var sr = b.AddComponent<SpriteRenderer>();
        sr.sprite = GameSprites.Get();
        sr.color = Color.yellow;
        b.transform.localScale = new Vector3(0.45f, 0.18f, 1f);
        barrel = b.transform;
        barrel.position = transform.position;
    }

    void Update() {
        float h = Input.GetAxisRaw("Horizontal");
        float v = Input.GetAxisRaw("Vertical");
        bool locked = Input.GetKey(KeyCode.LeftShift);

        if (h != 0f) faceDir = Mathf.Sign(h);
        if (locked) {
            rb.linearVelocity = new Vector2(0f, rb.linearVelocity.y);
        } else {
            rb.linearVelocity = new Vector2(h * runSpeed, rb.linearVelocity.y);
        }

        grounded = CheckGrounded();
        if (Input.GetKeyDown(KeyCode.Space) && grounded)
            rb.linearVelocity = new Vector2(rb.linearVelocity.x, jumpSpeed);

        Vector2 a = new Vector2(h, v);
        if (a.sqrMagnitude < 0.01f) a = new Vector2(faceDir, 0f);
        if (grounded && !locked && v < 0f && h == 0f) a = new Vector2(faceDir, 0f);
        aimDir = a.normalized;

        float ang = Mathf.Atan2(aimDir.y, aimDir.x) * Mathf.Rad2Deg;
        barrel.position = transform.position + (Vector3)(aimDir * barrelDist);
        barrel.rotation = Quaternion.Euler(0f, 0f, ang);

        fireTimer -= Time.deltaTime;
        bool fireHeld = Input.GetKey(KeyCode.LeftControl) || Input.GetKey(KeyCode.Z);
        if (fireHeld && fireTimer <= 0f) {
            fireTimer = fireCooldown;
            Fire();
        }
    }

    bool CheckGrounded() {
        Vector2 p = transform.position;
        var hits = Physics2D.OverlapBoxAll(p + Vector2.down * 1.05f, new Vector2(0.7f, 0.16f), 0f);
        foreach (var c in hits) {
            if (c == col || c.isTrigger) continue;
            return true;
        }
        return false;
    }

    void Fire() {
        var b = new GameObject("Bullet");
        b.transform.position = barrel.position + (Vector3)(aimDir * 0.25f);
        var sr = b.AddComponent<SpriteRenderer>();
        sr.sprite = GameSprites.Get();
        sr.color = Color.white;
        b.transform.localScale = new Vector3(0.25f, 0.25f, 1f);
        var bc = b.AddComponent<BoxCollider2D>();
        bc.isTrigger = true;
        bc.size = Vector2.one;
        var brb = b.AddComponent<Rigidbody2D>();
        brb.gravityScale = 0f;
        brb.linearVelocity = aimDir * bulletSpeed;
        var bl = b.AddComponent<Bullet2D>();
        bl.ignore = col;
    }
}
'''
CONTRA_FILES['FollowCam.cs'] = '''using UnityEngine;

public class FollowCam : MonoBehaviour {
    public Transform target;
    public float minX = -26f;
    public float maxX = 26f;
    void LateUpdate() {
        if (target == null) return;
        Vector3 p = transform.position;
        p.x = Mathf.Clamp(target.position.x, minX, maxX);
        transform.position = p;
    }
}
'''
UNITY = open('/content/unity_bin.txt').read().strip()
WORK = '/content/work'
t0 = time.time()
os.makedirs(WORK, exist_ok=True)
proj_candidates = sorted(glob.glob(cfg['DRIVE_PROJECTS'] + '/*/Assets')) if os.path.isdir(cfg['DRIVE_PROJECTS']) else []
if cfg['SYNC_PROJECT'] and proj_candidates:
    src_assets_parent = os.path.dirname(proj_candidates[0])
    name = os.path.basename(src_assets_parent)
    DST = os.path.join(WORK, name)
    r = subprocess.run(['bash', '-lc', 'mkdir -p "%s" && cp -a "%s/." "%s/"' % (WORK, src_assets_parent, DST)],
                       capture_output=True, text=True)
    print('synced project:', name, 'rc:', r.returncode, '(Drive-mount copies are slow for big assets)')
    if r.returncode != 0: print((r.stderr or '')[-1200:])
    assert r.returncode == 0, 'project copy failed'
else:
    name = cfg['GAME']
    DST = os.path.join(WORK, name)
    if not os.path.isdir(DST + '/Assets'):
        r = subprocess.run(['bash', '-lc', '"%s" -batchmode -quit -nographics -createProject "%s" -logFile /content/create.log; echo EXIT_CODE=$?' % (UNITY, DST)],
                           capture_output=True, text=True, timeout=1800)
        okc = [l for l in (r.stdout or '').splitlines() if l.startswith('EXIT_CODE=')]
        logtxt = open('/content/create.log', errors='ignore').read()[-2000:] if os.path.exists('/content/create.log') else '(no create.log — editor never started)'
        print('demo project created:', okc, '| %.1f min' % ((time.time() - t0) / 60))
        assert os.path.isdir(DST + '/Assets'), 'demo create failed: ' + logtxt
os.makedirs(os.path.join(DST, 'Assets', 'Editor'), exist_ok=True)
with open(os.path.join(DST, 'Assets', 'Editor', 'BuildScript.cs'), 'w') as f:
    f.write(CONTRA_FILES['BuildScript.cs'])
DEMO_DIR = os.path.join(DST, 'Assets', 'Demo')
if name == 'demo':
    os.makedirs(DEMO_DIR, exist_ok=True)
    GAME_BOOT_CS = '''using UnityEngine;
public static class DemoBoot {
    [RuntimeInitializeOnLoadMethod(RuntimeInitializeLoadType.AfterSceneLoad)]
    static void Boot() {
        var stale = GameObject.Find("Cube");
        if (stale != null) Object.Destroy(stale);
        if (GameObject.Find("CubeDodger") != null) return;
        new GameObject("CubeDodger").AddComponent<GameManager>();
        Debug.Log("GAME_BOOTED");
    }
}
'''
    open(os.path.join(DEMO_DIR, 'DemoBoot.cs'), 'w').write(GAME_BOOT_CS)
    GAME_MANAGER_CS = '''
using UnityEngine;
using System.Collections.Generic;

public class GameManager : MonoBehaviour {
    public static GameManager Instance { get; private set; }
    Transform ptx; Vector3 p0, cam0;
    List<GameObject> cubes = new List<GameObject>();
    List<float> spins = new List<float>();
    GameObject dashAnchor; float dashZ;
    float spawnT = 0.8f, speed = 7f, score, best, flashT, deadT;
    bool alive = true, audioReady;
    AudioClip thud, tick;

    Material Unlit(Color c) {
        var m = new Material(Shader.Find("Unlit/Color"));
        m.color = c;
        return m;
    }

    void Start() {
        Instance = this;
        var ground = GameObject.CreatePrimitive(PrimitiveType.Plane);
        ground.name = "Ground";
        ground.transform.localScale = new Vector3(6f, 1f, 60f);
        ground.GetComponent<Renderer>().material = Unlit(new Color(0.16f, 0.18f, 0.22f));
        dashAnchor = new GameObject("Dashes");
        for (float z = 0f; z < 120f; z += 4f) Dash(z);
        var cap = GameObject.CreatePrimitive(PrimitiveType.Capsule);
        cap.name = "Player";
        cap.GetComponent<Renderer>().material = Unlit(new Color(0.95f, 0.85f, 0.2f));
        p0 = new Vector3(0f, 1f, 0f);
        ptx = cap.transform;
        ptx.position = p0;
        cam0 = new Vector3(0f, 7.5f, p0.z - 10f);
        thud = MakeBeep(110f, 0.30f);
        tick = MakeBeep(300f, 0.06f);
        Debug.Log("GAME_READY");
    }

    void Dash(float z) {
        for (int k = -1; k <= 1; k += 2) {
            var d = GameObject.CreatePrimitive(PrimitiveType.Cube);
            d.name = "Dash";
            d.transform.SetParent(dashAnchor.transform, false);
            d.transform.localScale = new Vector3(0.25f, 0.05f, 1.6f);
            d.transform.position = new Vector3(4.5f * k, 0.05f, z);
            d.GetComponent<Renderer>().material = Unlit(new Color(0.35f, 0.38f, 0.45f));
        }
    }

    void Update() {
        if (flashT > 0f) flashT = Mathf.Max(0f, flashT - Time.deltaTime * 2f);
        if (!alive) {
            deadT += Time.deltaTime;
            if (deadT > 0.6f && (Input.anyKeyDown || Input.GetMouseButtonDown(0))) Restart();
            return;
        }
        speed = 7f + score * 0.05f;
        Vector3 pos = ptx.position;
        pos.z += speed * Time.deltaTime;
        float mv = Input.GetAxisRaw("Horizontal");
        if (Input.touchCount > 0 && Input.GetTouch(0).phase == TouchPhase.Began)
            mv = (Input.GetTouch(0).position.x >= Screen.width / 2) ? 1f : -1f;
        else if (Input.GetMouseButtonDown(0))
            mv = (Input.mousePosition.x >= Screen.width / 2) ? 1f : -1f;
        if (!audioReady && Mathf.Abs(mv) > 0.1f) {
            audioReady = true;
            if (tick != null) AudioSource.PlayClipAtPoint(tick, pos);
        }
        pos.x = Mathf.Clamp(pos.x + mv * 9f * Time.deltaTime, -9f, 9f);
        ptx.position = pos;
        score = pos.z;
        while (dashZ < pos.z + 120f) { Dash(dashZ); dashZ += 4f; }
        spawnT -= Time.deltaTime;
        if (spawnT <= 0) { spawnT = Mathf.Max(0.3f, 1.1f - score * 0.004f); Spawn(pos.z); }
        float fall = 4.5f + score * 0.01f;
        for (int i = cubes.Count - 1; i >= 0; i--) {
            var t = cubes[i].transform;
            float s = t.localScale.x;
            t.position += Vector3.down * fall * Time.deltaTime;
            t.Rotate(0f, spins[i] * Time.deltaTime, spins[i] * 0.6f * Time.deltaTime);
            if (t.position.y < -3f || t.position.z < pos.z - 12f) { Destroy(cubes[i]); cubes.RemoveAt(i); spins.RemoveAt(i); continue; }
            bool overlap = Mathf.Abs(t.position.x - pos.x) < 0.5f + 0.5f * s
                        && Mathf.Abs(t.position.z - pos.z) < 0.5f + 0.5f * s
                        && (t.position.y - 0.5f * s) < 2.0f && (t.position.y + 0.5f * s) > 0.0f;
            if (overlap) Die();
        }
        var cam = Camera.main.transform;
        cam.position = new Vector3(pos.x * 0.25f, 7.5f + speed * 0.02f, pos.z - 10f);
        cam.LookAt(pos + Vector3.forward * 6f);
    }

    void Spawn(float z) {
        var g = GameObject.CreatePrimitive(PrimitiveType.Cube);
        g.name = "Hazard";
        float s = Random.Range(1.0f, 1.6f);
        g.transform.localScale = new Vector3(s, s, s);
        float x = Random.Range(-9f, 9f);
        if (Random.value < 0.75f) {
            float lead = Mathf.Max(1.6f, 6f - score * 0.05f);
            float aim = ptx.position.x + Random.Range(-lead, lead);
            if (Mathf.Abs(aim - ptx.position.x) < 1.1f)
                aim = ptx.position.x + Mathf.Sign(aim - ptx.position.x + 0.001f) * 1.6f;
            x = Mathf.Clamp(aim, -9f, 9f);
        }
        g.transform.position = new Vector3(x, 13f, z + Random.Range(40f, 60f));
        g.GetComponent<Renderer>().material = Unlit(Random.value < 0.75f
            ? new Color(0.85f, 0.12f, 0.10f)
            : new Color(0.15f, 0.55f, 0.95f));
        cubes.Add(g);
        spins.Add(Random.Range(40f, 140f) * (Random.value < 0.5f ? 1f : -1f));
    }

    void Die() {
        if (!alive) return;
        alive = false; deadT = 0f; flashT = 1f;
        best = Mathf.Max(best, score);
        if (thud != null) AudioSource.PlayClipAtPoint(thud, new Vector3(0f, 2f, 0f));
        Debug.Log("GAME_DIED " + score.ToString("0") + "m");
    }

    void Restart() {
        alive = true; score = 0; spawnT = 0.8f; speed = 7f;
        foreach (var c in cubes) Destroy(c);
        cubes.Clear(); spins.Clear();
        ptx.position = p0;
        Camera.main.transform.position = cam0;
        Debug.Log("GAME_RESTART");
    }

    void OnGUI() {
        if (flashT > 0f) {
            GUI.color = new Color(1f, 0.1f, 0f, flashT * 0.55f);
            GUI.DrawTexture(new Rect(0, 0, Screen.width, Screen.height), Texture2D.whiteTexture);
            GUI.color = Color.white;
        }
        if (alive) {
            GUIStyle hud = new GUIStyle(GUI.skin.label) { fontSize = 20 };
            GUI.Label(new Rect(10, 8, 460, 32), "Distance: " + score.ToString("0") + " m" + (best > 0 ? "   Best: " + best.ToString("0") : ""), hud);
            if (score < 12f) {
                GUIStyle tip = new GUIStyle(GUI.skin.label) { fontSize = 14 };
                GUI.Label(new Rect(10, 40, 460, 24), "A/D or arrows dodge cubes; tap left/right half on touch", tip);
            }
        } else {
            GUI.Box(new Rect(Screen.width / 2 - 210, Screen.height / 2 - 90, 420, 180), "");
            var st = new GUIStyle(GUI.skin.label) { alignment = TextAnchor.MiddleCenter, wordWrap = true, fontSize = 18 };
            float left = Screen.width / 2 - 190, midY = Screen.height / 2;
            GUI.Label(new Rect(left, midY - 70, 380, 30), "GAME OVER", st);
            GUI.Label(new Rect(left, midY - 40, 380, 26), "Distance: " + score.ToString("0") + " m", st);
            GUI.Label(new Rect(left, midY - 14, 380, 26), "Best: " + best.ToString("0") + " m", st);
            GUI.Label(new Rect(left, midY + 14, 380, 26), deadT < 0.6f ? "..." : "Click, tap or press any key to restart", st);
        }
    }

    AudioClip MakeBeep(float freq, float dur) {
        int f = 44100, n = (int)(f * dur);
        var d = new float[n];
        for (int i = 0; i < n; i++) {
            float env = 1f - (float)i / n;
            d[i] = 0.5f * Mathf.Cos(6.28318f * freq * i / f) * env * env;
        }
        var clip = AudioClip.Create("beep", n, 1, f, false);
        clip.SetData(d, 0);
        return clip;
    }
}
'''
    open(os.path.join(DEMO_DIR, 'GameManager.cs'), 'w').write(GAME_MANAGER_CS)
    print('DEMO GAME WRITTEN: DemoBoot.cs + GameManager.cs ->', DEMO_DIR)
elif name == 'runandgun':
    cdir = os.path.join(DST, 'Assets', 'Contra')
    os.makedirs(cdir, exist_ok=True)
    for fn in sorted(CONTRA_FILES):
        if fn == 'BuildScript.cs':
            continue
        open(os.path.join(cdir, fn), 'w').write(CONTRA_FILES[fn])
    print('CONTRA SCRIPTS WRITTEN ->', cdir)
open('/content/work_project.txt', 'w').write(DST)
open('/content/work_name.txt', 'w').write(name)
print('GATE 3 PASS — project ready at', DST)


### 🏗️ 7 · Build + publish
**What this does:** builds for the form target (WebGL/Linux) with timing + RAM stats, then copies artifacts to Drive. Installs the WebGL module automatically if missing.

In [ ]:
import subprocess, os, json, time, glob, re, shutil
# --- ensure WebGL module (fresh VMs only; Drive-cached like the editor) then build + publish --- GATE 4 ---
cfg = json.load(open('/content/unitylab_config.json'))
UNITY = open('/content/unity_bin.txt').read().strip()
if not os.path.exists('/content/work_project.txt'):
    raise AssertionError('cell 6 (project sync) did not complete — run cell 6 first (its error will name the real cause)')
DST = open('/content/work_project.txt').read().strip()
t0 = time.time()

# prologue: clean stray entries a previous failed pkg install may have dumped in PlaybackEngines root
PEMB = '/content/ue/Editor/Data/PlaybackEngines'
for stray in ['Bee', 'UnityEditor.WindowsStandalone.Extensions.dll', 'UnityEditor.WindowsStandalone.Extensions.pdb',
              'WinPlayerBuildProgram.exe', 'WinPlayerBuildProgram.pdb', 'WinPlayerBuildProgram.Data.dll', 'WinPlayerBuildProgram.Data.pdb']:
    p = os.path.join(PEMB, stray)
    if os.path.isdir(p):
        shutil.rmtree(p, ignore_errors=True); print('cleaned stray dir:', stray)
    elif os.path.exists(p):
        os.remove(p); print('cleaned stray file:', stray)

import hashlib
def md5ok(path):
    m = hashlib.md5()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(1 << 22), b''):
            m.update(chunk)
    return m.hexdigest()

def fetch_cached(name, url, md5, cache_dir, timeout=3600):
    local = '/content/' + name
    cached = os.path.join(cache_dir, name)
    def have(p):
        return os.path.exists(p) and (not md5 or md5ok(p) == md5)
    if have(cached) and not have(local):
        shutil.copyfile(cached, local)
        print('%s from Drive cache (md5 OK)' % name if md5 else '%s from Drive cache' % name)
    if not have(local):
        if os.path.exists(local):
            print('stale/corrupt local %s removed' % name)
            os.remove(local)
        r = subprocess.run(['bash', '-lc', 'aria2c --allow-overwrite=true -x8 -d /content -o "%s" "%s"' % (name, url)],
                           capture_output=True, text=True, timeout=timeout)
        print('download %s rc:' % name, r.returncode)
        assert r.returncode == 0, '%s download failed: %s' % (name, (r.stderr or '')[-800:])
    if not have(local):
        bad = 'md5 mismatch on %s (local + cache)' % name
        print(bad)
        raise AssertionError(bad + ' - delete bad copies and rerun')
    if not have(cached):
        try:
            shutil.copyfile(local, cached)
            print('stashed %s to Drive cache' % name)
        except Exception as e:
            print('cache stash best-effort failed (fine):', repr(e))
    return local

def ensure_pkg(name, url, md5, check_dir, sig):
    if os.path.isfile(os.path.join(check_dir, sig)):
        print('%s module already present' % name)
        return
    import subprocess as sp
    for tool in ['7zip', 'cpio', 'xz-utils']:
        rr = sp.run(['bash', '-lc', 'apt-get install -y -qq %s >/dev/null 2>&1; echo APT_%s=$?' % (tool, tool.lower().replace('-', '_'))],
                    capture_output=True, text=True)
        assert ('=0' in rr.stdout) or ('=7' in rr.stdout and 'newest' in rr.stdout), 'apt ' + tool + ' failed: ' + (rr.stderr or '')[-600:]
    local = fetch_cached(name, url, md5, cfg['DRIVE_CACHE'])
    U1, U2 = '/content/pkgx1', '/content/pkgx2'
    shutil.rmtree(U1, ignore_errors=True); shutil.rmtree(U2, ignore_errors=True)
    os.makedirs(U1); os.makedirs(U2)
    r = subprocess.run(['bash', '-lc', '7z x -y "%s" -o"%s" >/dev/null' % (local, U1)],
                       capture_output=True, text=True, timeout=3600)
    print('7z pkg rc:', r.returncode)
    if r.returncode != 0:
        print((r.stderr or r.stdout or '')[-1500:])
        raise AssertionError('7z pkg failed — paste output')
    cands = [f for f in os.listdir(U1) if f.lower().startswith('payload')]
    assert cands, 'no Payload* in pkg: ' + str(sorted(os.listdir(U1)))
    pay = os.path.join(U1, cands[0])
    r = subprocess.run(['bash', '-lc', '7z x -y "%s" -o"%s" >/dev/null' % (pay, U2)],
                       capture_output=True, text=True, timeout=3600)
    print('7z payload rc:', r.returncode)
    if r.returncode != 0:
        print((r.stderr or r.stdout or '')[-1500:])
        raise AssertionError('7z payload failed — paste output')
    # generic placement: descend the single-child chain until the dir name matches
    # the destination basename OR the root has multiple entries (payload root = module dir)
    src = U2
    while True:
        ents = os.listdir(src)
        if len(ents) == 1 and os.path.isdir(os.path.join(src, ents[0])) and ents[0] != os.path.basename(check_dir):
            src = os.path.join(src, ents[0])
            continue
        break
    os.makedirs(check_dir, exist_ok=True)
    r2 = subprocess.run(['bash', '-lc', 'cp -a "%s/." "%s/"' % (src, check_dir)], capture_output=True, text=True)
    assert r2.returncode == 0, 'cp -a module install failed: ' + (r2.stderr or '')[-1200:]
    assert os.path.isdir(check_dir) and len(os.listdir(check_dir)) > 0, 'module destination empty after install'
    print('MODULE_INSTALLED:', check_dir)

PEMB = '/content/ue/Editor/Data/PlaybackEngines'
if cfg['BUILD_TARGET'] == 'WebGL' and not os.path.isdir(PEMB + '/WebGLSupport'):
    local = fetch_cached('WebGL.tar.xz', cfg['WEBGL_URL'], cfg['WEBGL_MD5'], cfg['DRIVE_CACHE'])
    r = subprocess.run(['bash', '-lc', 'tar -xJf "%s" -C /content/ue && echo EXTRACT_OK' % local],
                       capture_output=True, text=True, timeout=900)
    print('WebGL module extract:', r.returncode)
    assert os.path.isdir(PEMB + '/WebGLSupport'), 'WebGLSupport missing - delete cached tarball if corrupt and rerun'
    print('MODULE_INSTALLED: WebGLSupport')

if cfg['BUILD_TARGET'] == 'Windows':
    ensure_pkg('Windows.pkg', cfg['WINDOWS_URL'], cfg['WINDOWS_MD5'], PEMB + '/WindowsStandaloneSupport', 'UnityEditor.WindowsStandalone.Extensions.dll')
    print('MEASURE target=Windows')

if cfg['BUILD_TARGET'] == 'Mac':
    ensure_pkg('Mac.pkg', cfg['MAC_URL'], cfg['MAC_MD5'], PEMB + '/MacStandaloneSupport', 'UnityEditor.MacStandalone.Extensions.dll')
    print('MEASURE target=Mac')

if cfg['BUILD_TARGET'] == 'Android':
    ensure_pkg('Android.pkg', cfg['ANDROID_URL'], cfg['ANDROID_MD5'], PEMB + '/AndroidPlayer', 'UnityEditor.Android.Extensions.dll')
    CH = PEMB + '/AndroidPlayer'
    NDKP = CH + '/NDK'
    if os.path.isdir(NDKP):
        rev = ''
        spp = os.path.join(NDKP, 'source.properties')
        if os.path.exists(spp):
            for ln in open(spp, errors='ignore'):
                if ln.startswith('Pkg.Revision'):
                    rev = ln.split('=')[-1].strip()
        if not rev.startswith('27.2'):
            print('NDK wrong version (%s) - wiping; Unity 6000.3 needs r27c' % (rev or 'unknown'))
            subprocess.run(['bash', '-lc', 'rm -rf "%s"' % NDKP], capture_output=True)
    for [n, u, md5e, dest, rn_from, rn_to, check] in cfg['ANDROID_CHAIN']:
        if os.path.isdir(os.path.join(PEMB, check)):
            print('chain %s already installed' % n)
            continue
        destabs = os.path.join(PEMB, dest)
        local = fetch_cached('and_' + n + '.zip', u, md5e or '', cfg['DRIVE_CACHE'])
        U = '/content/andx'
        shutil.rmtree(U, ignore_errors=True)
        os.makedirs(U, exist_ok=True)
        r = subprocess.run(['bash', '-lc', 'unzip -q -o "%s" -d "%s"' % (local, U)],
                           capture_output=True, text=True, timeout=3600)
        assert r.returncode == 0, 'unzip %s failed: %s' % (n, (r.stderr or '')[-1200:])
        if rn_from:
            # wrapped zips carry a <rn_from>/ dir; some (cmake) ship payload at zip root
            wrapped = os.path.isdir(os.path.join(U, rn_from))
            target = os.path.join(destabs, rn_to)
            os.makedirs(target, exist_ok=True)
            src2 = os.path.join(U, rn_from) if wrapped else U
            r2 = subprocess.run(['bash', '-lc', 'cp -a "%s/." "%s/"' % (src2, target)], capture_output=True, text=True)
            assert r2.returncode == 0, 'cp -a %s failed: ' % n + (r2.stderr or '')[-1200:]
        else:
            os.makedirs(destabs, exist_ok=True)
            r2 = subprocess.run(['bash', '-lc', 'cp -a "%s/." "%s/"' % (U, destabs)], capture_output=True, text=True)
            assert r2.returncode == 0, 'cp -a %s failed: ' % n + (r2.stderr or '')[-1200:]
        assert os.path.isdir(os.path.join(PEMB, check)), 'install check failed for %s' % n
        print('chain %s installed' % n)
    src = CH + '/OpenJDK'
    if not os.path.isdir(src):
        local = fetch_cached('openjdk.zip', cfg['ANDROID_JDK_URL'], '', cfg['DRIVE_CACHE'])
        U = '/content/jdkx'
        shutil.rmtree(U, ignore_errors=True)
        os.makedirs(U, exist_ok=True)
        r = subprocess.run(['bash', '-lc', 'unzip -q -o "%s" -d "%s"' % (local, U)],
                           capture_output=True, text=True, timeout=3600)
        assert r.returncode == 0, 'unzip openjdk failed: ' + (r.stderr or '')[-1200:]
        subs = os.listdir(U)
        srcu = (U + '/' + subs[0]) if len(subs) == 1 else U
        os.makedirs(src, exist_ok=True)
        r2 = subprocess.run(['bash', '-lc', 'cp -a "%s/." "%s/"' % (srcu, src)], capture_output=True, text=True)
        assert r2.returncode == 0, 'cp -a openjdk failed: ' + (r2.stderr or '')[-1200:]
    assert os.path.isdir(PEMB + '/AndroidPlayer/SDK') and os.path.isdir(PEMB + '/AndroidPlayer/NDK')         and os.path.isdir(PEMB + '/AndroidPlayer/OpenJDK'), 'AndroidPlayer SDK/NDK/OpenJDK incomplete'
    print('MEASURE target=Android')

shutil.rmtree(os.path.join(DST, 'Temp'), ignore_errors=True)
print('TEMP_CLEARED (pre-build hygiene)')
NAME = open('/content/work_name.txt').read().strip() if os.path.exists('/content/work_name.txt') else 'demo'
LIB = os.path.join(DST, 'Library')
LIBTGZ = os.path.join(cfg['DRIVE_CACHE'], 'library-' + NAME + '.tar.gz') if cfg['ENABLE_DRIVE_CACHE'] else None
if LIBTGZ and os.path.exists(LIBTGZ) and not os.path.isdir(LIB):
    tr = subprocess.run(['bash', '-lc', 'mkdir -p "%s" && tar -xzf "%s" -C "%s"' % (DST, LIBTGZ, DST)], capture_output=True, text=True, timeout=1800)
    print('Library cache restore rc:', tr.returncode)
    if tr.returncode != 0: print((tr.stderr or '')[-1200:])

CS = os.path.join(DST, 'Assets', 'Editor', 'BuildScript.cs')
cs = open(CS).read()
BT_MAP = {'Linux': 'StandaloneLinux64', 'WebGL': 'WebGL', 'Windows': 'StandaloneWindows64', 'Android': 'Android', 'Mac': 'StandaloneOSX'}
cs = cs.replace('var target = BuildTarget.WebGL; var outDir = "Builds/WebGL/probe";',
                ('var target = BuildTarget.%s; var outDir = "Builds/%s/probe";' % (BT_MAP[cfg['BUILD_TARGET']], cfg['BUILD_TARGET'])))
if cfg['BUILD_TARGET'] == 'Android':
    # official headless tool-path API (lives in the Android.Extensions module we install)
    PEMA = '/content/ue/Editor/Data/PlaybackEngines'
    NLB_INJ = '\n'
    cs = cs.replace('    public static void PerformBuild() {',
        ('    public static void PerformBuild() {' + NLB_INJ + 
        '        UnityEditor.Android.AndroidExternalToolsSettings.sdkRootPath = "%s/AndroidPlayer/SDK";' + NLB_INJ + 
        '        UnityEditor.Android.AndroidExternalToolsSettings.ndkRootPath = "%s/AndroidPlayer/NDK";' + NLB_INJ + 
        '        UnityEditor.Android.AndroidExternalToolsSettings.jdkRootPath = "%s/AndroidPlayer/OpenJDK";' + NLB_INJ + 
        '        UnityEditor.Android.AndroidExternalToolsSettings.maxJvmHeapSize = 2048;' + NLB_INJ + 
        '        Debug.Log("ANDROID_TOOLS_SET sdk=" + UnityEditor.Android.AndroidExternalToolsSettings.sdkRootPath + " ndk=" + UnityEditor.Android.AndroidExternalToolsSettings.ndkRootPath + " jdk=" + UnityEditor.Android.AndroidExternalToolsSettings.jdkRootPath);')
        % (PEMA, PEMA, PEMA))
open(CS, 'w').write(cs)

SAVE_TAG = cfg['BUILD_TARGET']
LOG = '/content/build_%s.log' % SAVE_TAG.lower()
TF = '/content/time_%s.txt' % SAVE_TAG.lower()
JAVAEXP = ('export JAVA_HOME="/content/ue/Editor/Data/PlaybackEngines/AndroidPlayer/OpenJDK"; export PATH="$JAVA_HOME/bin:$PATH"; '
          if cfg['BUILD_TARGET'] == 'Android' else '')
BUILD_CMD = ('export XDG_RUNTIME_DIR=/tmp/xdg; ' + JAVAEXP + ' cd "%s" && /usr/bin/time -v "%s" '
             '-batchmode -nographics -quit -projectPath "%s" -executeMethod BuildScript.PerformBuild '
             '-logFile %s 2> %s; echo EXIT_CODE=$?' % (DST, UNITY, DST, LOG, TF))
try:
    r = subprocess.run(['bash', '-lc', BUILD_CMD], capture_output=True, text=True, timeout=2700)
except Exception as e:
    print('RUNTIME_TIMEOUT_OR_ERROR:', repr(e))
    lg = open(LOG, errors='ignore').read() if os.path.exists(LOG) else '(no log)'
    print('\n'.join(lg.splitlines()[-50:])); raise
print('BASH_RETURN_CODE:', r.returncode)
peak = None
if os.path.exists(TF):
    for line in open(TF, errors='ignore'):
        if line.startswith(('Elapsed (wall clock)', 'Maximum resident', 'Percent of CPU', 'Exit status')):
            print('MEASURE:', line.strip())
    m = re.search(r'Maximum resident set size.*?: (\d+)', open(TF, errors='ignore').read())
    peak = int(m.group(1)) if m else None
log = open(LOG, errors='ignore').read() if os.path.exists(LOG) else '(no log)'
ok = 'BUILD_OK' in log
print('exit:', r.returncode, '| wall: %.1f min' % ((time.time() - t0) / 60), '| peak RAM: %.2f GB' % ((peak or 0) / 1048576.0), '| BUILD_OK:', ok)
out_dir = os.path.join(DST, 'Builds', SAVE_TAG)
arts = sorted(glob.glob(out_dir + '/*'))
print('artifacts:', len(arts), 'entries in', out_dir)
if not ok:
    print('log tail:'); print('\n'.join(log.splitlines()[-50:]))
assert ok, 'BUILD FAILED'

if SAVE_TAG == 'Windows':
    bare = os.path.join(out_dir, 'probe')
    if os.path.exists(bare) and not os.path.exists(bare + '.exe'):
        os.rename(bare, bare + '.exe')
        print('RENAMED extensionless Windows exe -> probe.exe (Linux-cross-build quirk)')

stamp = time.strftime('%Y%m%d-%H%M%S')
dest = os.path.join(cfg['DRIVE_BUILDS'], stamp + '-' + SAVE_TAG)
os.makedirs(dest, exist_ok=True)
r = subprocess.run(['bash', '-lc', 'cp -a "%s" "%s/"' % (out_dir, dest)], capture_output=True, text=True)
print('PUBLISH rc:', r.returncode, '->', dest)
if r.returncode != 0: print((r.stderr or '')[-1200:])
n_pub = len(glob.glob(dest + '/**', recursive=True))
print('PUBLISHED to Drive:', n_pub, 'files at', dest)
if LIB and os.path.isdir(LIB):
    if LIBTGZ:
        try:
            tl = time.time()
            subprocess.run(['bash', '-lc', 'tar -czf "%s" -C "%s" Library' % (LIBTGZ, DST)], timeout=1800)
            print('Library persisted: %.2f GB in %.1f s' % (os.path.getsize(LIBTGZ) / 1073741824.0, time.time() - tl))
        except Exception as e:
            print('Library persist best-effort failed (fine):', repr(e))
print('GATE 4 PASS — %s build' % SAVE_TAG)


### 📋 8 · Report
**What this does:** prints the full system state — paste this cell's output back to the agent.

In [ ]:
import subprocess, os, json, glob
# --- system report — paste this whole output back to the agent ---
cfg = json.load(open('/content/unitylab_config.json'))
print('===== unity-lab v0 SYSTEM REPORT =====')
print('EDITOR:', cfg['VERSION'], '| rev:', cfg['REV'])
ubin = '/content/unity_bin.txt'
print('unity_bin:', open(ubin).read().strip() if os.path.exists(ubin) else '(boot cell did not finish)')
lic = os.path.expanduser('~/.config/unity3d/Unity/licenses/UnityEntitlementLicense.xml')
print('license:', 'ACTIVE' if os.path.exists(lic) else 'MISSING')
wproj = '/content/work_project.txt'
print('project:', open(wproj).read().strip() if os.path.exists(wproj) else '(cell 6 did not finish)')
print('form: cache=%s target=%s' % (cfg['ENABLE_DRIVE_CACHE'], cfg['BUILD_TARGET']))
print('local builds:', sorted(glob.glob('/content/work/*/Builds/*/*')))
print('drive builds:', sorted(glob.glob(cfg['DRIVE_BUILDS'] + '/*'))[-5:])
print('drive cache:', sorted(glob.glob(cfg['DRIVE_CACHE'] + '/*')) if os.path.isdir(cfg['DRIVE_CACHE']) else '(cache off)')
print('DISK:', subprocess.run(['bash', '-lc', 'df -h /content | tail -1'], capture_output=True, text=True).stdout.strip())


### 🌉 9 · Agent bridge
**What this does:** installs + starts the Drive job-bus runner (idempotent). After this cell the agent can drive this VM from the PC with zero pastes.

In [ ]:
# === agent bridge: install + start job-bus runner (idempotent) ===
import subprocess, os, time, glob, json
RUNNER_NB = '''#!/usr/bin/env python3
# runner.py — unity-lab Drive job-bus runner (VM side). Canonical copy: bus-jobs/runner.py
# Fetched live from /content/runner.py via the bus itself (file_get + line-range jobs).
# Embedded into unity-lab.ipynb bridge cell by the builder (P3.5).
import os, sys, json, time, socket, hashlib, subprocess, glob, shutil

BUS = '/content/drive/MyDrive/unity-lab/bus'
IN, CLAIMED, OUT, LOGS = [os.path.join(BUS, d) for d in ('in', 'claimed', 'out', 'logs')]
VERSION, POLL = 'runner-v1', 10

ERR = open('/content/bus_runner.err', 'a', buffering=1)
sys.stdout = open('/content/bus_runner.log', 'a', buffering=1)
sys.stderr = ERR


def fp():
    return hashlib.sha1((socket.gethostname() + ':unity-lab').encode()).hexdigest()[:12]


def tail(s, n=1500):
    s = s or ''
    return s[-n:]


def write_json(path, obj):
    # Drive-fuse replace-path does not re-upload same-name replaces reliably;
    # delete-then-write (new file) propagates. Transient absence is fine: poll retries.
    if os.path.exists(path):
        os.remove(path)
    with open(path, 'w') as f:
        json.dump(obj, f)


def heartbeat(busy):
    hb = {'version': VERSION, 'fp': fp(), 'ts': time.time(), 'busy': busy,
          'paused': os.path.exists('/content/bus_paused'),
          'disk_gb': round(os.statvfs('/content').f_bavail * os.statvfs('/content').f_frsize / 1e9, 1)}
    write_json(os.path.join(BUS, 'heartbeat.json'), hb)


def write_out(jid, payload):
    payload['version'] = VERSION
    write_json(os.path.join(OUT, jid + '.json'), payload)
    with open(os.path.join(BUS, 'audit.jsonl'), 'a') as f:
        f.write(json.dumps(payload) + chr(10))


def run_job(path):
    name = os.path.basename(path)
    claim = os.path.join(CLAIMED, name)
    try:
        os.rename(path, claim)
    except OSError:
        return
    try:
        job = json.load(open(claim, errors='ignore'))
    except Exception as e:
        write_out(name.replace('.json', ''), {'id': name, 'rc': 4, 'err': 'bad json %r' % e, 'ts': time.time()})
        return
    jid = str(job.get('id', name.replace('.json', '')))
    t0, typ, body = time.time(), str(job.get('type', 'shell')), str(job.get('body', ''))
    to = int(job.get('timeout', 600))
    res = {'id': jid, 'type': typ, 'ts': t0}
    try:
        if typ in ('shell', 'python'):
            cmd = ['bash', '-lc', body] if typ == 'shell' else [sys.executable, '-c', body]
            r = subprocess.run(cmd, capture_output=True, text=True, timeout=to)
            res.update(rc=r.returncode, out=tail(r.stdout), err=tail(r.stderr))
        elif typ == 'status':
            res.update(rc=0, out='RUNNER_ALIVE %s' % VERSION, err='')
        elif typ == 'file_get':
            p = body
            if os.path.exists(p):
                sz = os.path.getsize(p)
                if sz <= 200000:
                    res.update(rc=0, out=open(p, errors='ignore').read(), err='size=%d' % sz)
                else:
                    shutil.copyfile(p, os.path.join(LOGS, jid + '.bin'))
                    res.update(rc=0, out='TOO_BIG_SEE_LOGS/%s.bin' % jid, err='size=%d' % sz)
            else:
                res.update(rc=2, out='', err='no such file: %s' % p)
        elif typ == 'PAUSE':
            if body.strip() == 'off':
                os.remove('/content/bus_paused')
                res.update(rc=0, out='UNPAUSED', err='')
            else:
                open('/content/bus_paused', 'w').write('1')
                res.update(rc=0, out='PAUSED', err='')
        else:
            res.update(rc=3, out='', err='unknown type %s' % typ)
    except subprocess.TimeoutExpired as e:
        res.update(rc=124, out=tail((e.stdout or b'').decode(errors='ignore') if isinstance(e.stdout, bytes) else str(e.stdout or '')), err='TIMEOUT %ss' % to)
    except Exception as e:
        res.update(rc=1, out='', err='EXC %r' % e)
    res['wall'] = round(time.time() - t0, 1)
    write_out(jid, res)


busy = False
while True:
    try:
        if os.path.exists('/content/bus_paused'):
            heartbeat(busy)
        else:
            jobs = sorted(glob.glob(os.path.join(IN, '*.json')))
            busy = bool(jobs)
            heartbeat(busy)
            for j in jobs:
                busy = True
                run_job(j)
    except Exception as e:
        print('LOOP_EXC %r' % e, file=sys.stderr)
    time.sleep(POLL)
'''
BUS = '/content/drive/MyDrive/unity-lab/bus'
for d in ('in', 'claimed', 'out', 'logs'):
    os.makedirs(os.path.join(BUS, d), exist_ok=True)
with open('/content/runner.py', 'w') as f:
    f.write(RUNNER_NB)
_pc = subprocess.run(['python3', '-c', "import ast; ast.parse(open('/content/runner.py').read())"],
                     capture_output=True, text=True)
if _pc.returncode != 0:
    print('RUNNER_PARSE_FAIL:', (_pc.stderr or '')[-800:])
    raise SystemExit(1)
print('RUNNER_PARSE_OK')
alive = subprocess.run(['bash', '-lc', 'pgrep -f "[r]unner.py" || true'], capture_output=True, text=True).stdout.strip()
if alive:
    print('RUNNER_ALREADY_ALIVE:', alive.splitlines()[0])
else:
    _sp = subprocess.run(['bash', '-lc',
        'nohup setsid python3 /content/runner.py 1>>/content/bus_runner.log 2>>/content/bus_runner.err & echo SPAWNED_PID=$!'],
        capture_output=True, text=True)
    print('RUNNER_SPAWNED', _sp.stdout.strip(), (_sp.stderr or '')[-300:])
hb = None
for _ in range(10):
    time.sleep(2)
    try:
        hbp = os.path.join(BUS, 'heartbeat.json')
        if os.path.exists(hbp):
            hb = json.load(open(hbp))
            if time.time() - hb.get('ts', 0) < 30:
                break
            hb = None
    except Exception:
        pass
print('BRIDGE_OK:', json.dumps(hb) if hb else 'NO_FRESH_HEARTBEAT (check /content/bus_runner.err)')
